# 📞 OpenAI Tool Calling

## Learning Objectives
In this notebook, you will learn:
1. **Tool Calling Basics** - How OpenAI models decide when and which tools to invoke
2. **Defining Tools with `@tool`** - Creating LangChain tools using the decorator pattern
3. **Binding Tools to an LLM** - Using `bind_tools()` to make tools available to a model
4. **Inspecting Tool Calls** - Reading the structured tool call output from the model
5. **Manual Tool Execution Loop** - Building the full request → tool call → response cycle

## Prerequisites
- Basic understanding of LangChain and LLMs
- A `.env` file with `OPENAI_API_KEY` configured
- Packages from `pyproject.toml` installed
- Completed **Notebook 6.0** (Tools & Functions Essentials)

---

## 📖 What is Tool Calling?

OpenAI's **tool calling** (previously called "function calling") lets a model decide to invoke one or more tools based on a user's prompt. Instead of generating a text answer, the model returns a structured JSON payload describing which tool to call and with what arguments.

### Key Concepts:
- **Tool selection is automatic** — the model reads tool descriptions and picks the right one
- **Multiple tools can fire in parallel** — a single prompt can trigger several tool calls at once
- **The model doesn't execute tools** — your code runs the tool and feeds results back to the model

> **Note**: OpenAI originally called this "function calling", then renamed it to "tool calling", and the terminology continues to evolve. Functionally they are the same. See the [official docs](https://platform.openai.com/docs/guides/function-calling).

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and Configuration
# ============================================================================

import os
import sys
import warnings

warnings.filterwarnings("ignore")

sys.path.append(os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))

print("✅ Environment configured successfully!")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: OpenAI Chat Model
# ============================================================================

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---

## 🚫 Part 1: LLM Without Tools

Without tools, the model can only answer from its training data. It cannot access real-time information like weather, databases, or APIs. Let's see this limitation in action.

In [ ]:
# ============================================================================
# TEST: LLM Response Without Tools
# ============================================================================

response = llm.invoke("How will the weather be in Munich today?")

print(f"🤖 Response (no tools):\n{response.content}")

---

## 🔧 Part 2: Defining Custom Tools

We define tools using the `@tool` decorator from LangChain. Each tool is a Python function with a docstring — the model reads the docstring to understand when and how to use the tool.

> **Key Insight**: The quality of your docstring directly affects how well the model selects and uses the tool. Be specific about what the tool does, its arguments, and return values.

In [ ]:
# ============================================================================
# CUSTOM TOOLS: Define Weather and Seating Tools
# ============================================================================

from langchain_core.tools import tool


@tool
def fake_weather_api(city: str) -> str:
    """
    Check the weather in a specified city.

    Args:
        city (str): The name of the city where you want to check the weather.

    Returns:
        str: A description of the current weather in the specified city.
    """
    return "Sunny, 22°C"


@tool
def outdoor_seating_availability(city: str) -> str:
    """
    Check if outdoor seating is available at a specified restaurant in a given city.

    Args:
        city (str): The name of the city where you want to check for outdoor seating availability.

    Returns:
        str: A message stating whether outdoor seating is available or not.
    """
    return "Outdoor seating is available."


tools = [fake_weather_api, outdoor_seating_availability]

print(f"✅ Defined {len(tools)} tools: {[t.name for t in tools]}")

In [ ]:
# ============================================================================
# EXPLORE: Tool Properties and Direct Invocation
# ============================================================================

print(f"📋 Weather tool description:\n{fake_weather_api.description}")
print()
print(f"📋 Seating tool description:\n{outdoor_seating_availability.description}")

# --- Direct invocation (bypassing the LLM) ---
print(f"\n🔧 Direct call — Weather: {fake_weather_api.invoke({'city': 'Munich'})}")
print(f"🔧 Direct call — Seating: {outdoor_seating_availability.invoke({'city': 'Munich'})}")

---

## 🔗 Part 3: Binding Tools to the LLM

Calling `llm.bind_tools(tools)` creates a new LLM instance that knows about our tools. When invoked, the model inspects the tool descriptions and decides whether to call one, multiple, or none based on the user's query.

In [ ]:
# ============================================================================
# BIND TOOLS: Attach Tools to the LLM
# ============================================================================

llm_with_tools = llm.bind_tools(tools)

print("✅ Tools bound to LLM!")

### 🎯 Single Tool Call

When the query only relates to one tool, the model selects just that tool.

In [ ]:
# ============================================================================
# TEST: Single Tool Call
# ============================================================================

result = llm_with_tools.invoke("How will the weather be in Munich today?")

print(f"🔧 Tool calls: {result.tool_calls}")
print(f"🤖 Content: '{result.content}'")

### 🎯🎯 Multiple Tool Calls

When the query touches multiple tools, the model can fire them all in a single response — no extra round-trips needed.

In [ ]:
# ============================================================================
# TEST: Multiple Tool Calls in One Request
# ============================================================================

result = llm_with_tools.invoke(
    "How will the weather be in Munich today? Do you still have seats outdoor available?"
)

print(f"🔧 Number of tool calls: {len(result.tool_calls)}")
for tc in result.tool_calls:
    print(f"   → {tc['name']}({tc['args']})")

### 🔍 Inspecting the Tool Call Response

The model returns an `AIMessage` with empty `content` and populated `tool_calls`. Let's inspect the full structure.

In [ ]:
# ============================================================================
# INSPECT: Tool Call Structure
# ============================================================================

print("📋 tool_calls attribute:")
for tc in result.tool_calls:
    print(f"   name: {tc['name']}")
    print(f"   args: {tc['args']}")
    print(f"   id:   {tc['id']}")
    print()

print("📋 Full model_dump():")
result.model_dump()

---

## 🔄 Part 4: Manual Tool Execution Loop

The model doesn't execute tools itself — it only *requests* tool calls. Your code must:

1. Send the user message to the LLM
2. Read which tools the model wants to call
3. Execute each tool and collect results
4. Send everything back to the LLM for a final natural-language response

This is the core loop that agents automate under the hood.

In [ ]:
# ============================================================================
# STEP 1: Send User Message and Get Tool Call Requests
# ============================================================================

from langchain_core.messages import HumanMessage, ToolMessage

messages = [
    HumanMessage(
        "How will the weather be in Munich today? I would like to eat outside if possible"
    )
]

llm_output = llm_with_tools.invoke(messages)
messages.append(llm_output)

print(f"🤖 LLM requested {len(llm_output.tool_calls)} tool call(s):")
for tc in llm_output.tool_calls:
    print(f"   → {tc['name']}({tc['args']})")

In [ ]:
# ============================================================================
# STEP 2: Execute Tools and Append Results
# ============================================================================

tool_mapping = {
    "fake_weather_api": fake_weather_api,
    "outdoor_seating_availability": outdoor_seating_availability,
}

for tool_call in llm_output.tool_calls:
    tool_fn = tool_mapping[tool_call["name"].lower()]
    tool_output = tool_fn.invoke(tool_call["args"])
    messages.append(ToolMessage(tool_output, tool_call_id=tool_call["id"]))
    print(f"🔧 {tool_call['name']} → {tool_output}")

In [ ]:
# ============================================================================
# STEP 3: Get Final Response with Tool Results
# ============================================================================

final_response = llm_with_tools.invoke(messages)

print(f"🤖 Final Response:\n{final_response.content}")

In [ ]:
# ============================================================================
# INSPECT: Full Message History
# ============================================================================

print("📋 Complete message flow:")
for i, msg in enumerate(messages):
    print(f"   [{i}] {msg.__class__.__name__}: {msg.content[:80] or '(tool call request)'}")

---

## 📝 Summary

In this notebook, we learned:

### 1. Tool Calling Basics
- Without tools, LLMs cannot access real-time data
- **Tool calling** lets the model request external function execution via structured JSON

### 2. Defining and Binding Tools
- The `@tool` decorator turns any Python function into a LangChain tool
- **Docstrings matter** — the model uses them to decide when to call each tool
- `bind_tools()` attaches tools to an LLM instance

### 3. Single vs. Multiple Tool Calls
- The model can invoke **one or many tools** in a single response
- Each tool call includes a unique `id` for matching results back

### 4. Manual Tool Execution Loop
- The 3-step cycle: **User → LLM (tool requests) → Execute tools → LLM (final answer)**
- `ToolMessage` carries tool results back to the model, matched by `tool_call_id`
- This loop is what agents like `create_react_agent` automate internally

### Next Steps
- Return to **Notebook 6.2** to see how agents automate this loop
- Explore structured outputs with Pydantic models for more complex tool schemas
- Try building a multi-turn conversation with persistent tool state